# 02 - Vector Database

This notebook takes the chunks produced by `01_Document_Preprocessing.ipynb`,
embeds them with a HuggingFace sentence-embedding model, and builds a
persisted **ChromaDB** collection that Notebook 3 will query at answer time.

```
Processed Chunks -> [THIS NOTEBOOK: Embed + Index] -> ChromaDB (persisted) -> Retriever
```

> **Note:** Generating embeddings requires downloading the model
> `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace on first run,
> so an internet connection is needed the first time this notebook executes.
> Subsequent runs reuse the local HuggingFace cache.


## Imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from rag_core import AppConfig, DocumentProcessor, EmbeddingManager, VectorDatabaseManager

print(f"Project root: {PROJECT_ROOT}")


Project root: f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant


## Configuration

In [2]:
config = AppConfig()
config.project_root = PROJECT_ROOT
config.__post_init__()
config.ensure_directories()

print("Embedding model :", config.embedding_model_name)
print("Persist dir     :", config.persist_directory)
print("Collection name :", config.collection_name)


Embedding model : sentence-transformers/all-MiniLM-L6-v2
Persist dir     : f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db
Collection name : course_materials


## Load Processed Chunks

Chunks are loaded from the JSON Lines file written by Notebook 1, so no
document parsing happens again here.

In [3]:
processor = DocumentProcessor(config)
chunks = processor.load_chunks()

print(f"Loaded {len(chunks)} processed chunks.")
print("Example chunk id:", chunks[0].chunk_id)


2026-08-02 01:03:04 | INFO     | DocumentProcessor | Loaded 13 chunk(s) from f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db\processed_chunks.jsonl
Loaded 13 processed chunks.
Example chunk id: Artificial_Intelligence__expert_systems__p0__c00001


## Embedding Model

`EmbeddingManager` wraps `HuggingFaceEmbeddings` and lazily loads the model
on first access, so importing this notebook's cells does not immediately
trigger a download.

In [4]:
embedding_manager = EmbeddingManager(config)

# Loading the model here makes the download cost explicit and visible,
# rather than happening silently inside the next cell.
_ = embedding_manager.model
print("Embedding model loaded:", config.embedding_model_name)


2026-08-02 01:03:04 | INFO     | EmbeddingManager | Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' on 'cpu'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-08-02 01:03:09 | INFO     | EmbeddingManager | Embedding model ready.
Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


## Generate Embeddings

We embed a small preview batch first just to confirm the model returns
vectors of the expected dimensionality before embedding the full corpus
(which happens automatically inside `VectorDatabaseManager.build`).

In [5]:
preview_texts = [c.text for c in chunks[:3]]
preview_vectors = embedding_manager.embed_documents(preview_texts)

print(f"Embedded {len(preview_vectors)} preview chunks.")
print(f"Embedding dimensionality: {len(preview_vectors[0])}")


Embedded 3 preview chunks.
Embedding dimensionality: 384


## Create Chroma Database

`VectorDatabaseManager.build` embeds every chunk and persists the resulting
collection to `chroma_db/`. If a persisted database already exists, it is
loaded instead of being recomputed, per the "never recreate embeddings
unnecessarily" requirement — pass `overwrite=True` to force a rebuild.

In [6]:
vector_db_manager = VectorDatabaseManager(config, embedding_manager)

vector_store = vector_db_manager.build(chunks, overwrite=False)
print("Vectors stored:", vector_db_manager.count())


2026-08-02 01:03:13 | INFO     | VectorDatabaseManager | Embedding and indexing 13 chunk(s)...
2026-08-02 01:03:14 | INFO     | VectorDatabaseManager | Vector database persisted to f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db
Vectors stored: 13


## Persist Database

ChromaDB with a `persist_directory` writes to disk automatically as documents are added, so no separate flush call is required. We simply confirm the files exist on disk.

In [7]:
persisted_files = list(config.persist_directory.rglob("*"))
print(f"Files under {config.persist_directory}: {len(persisted_files)}")
for f in sorted(persisted_files)[:10]:
    print(" -", f.relative_to(config.persist_directory))


Files under f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db: 7
 - 1ababa8a-23b5-4db6-90c3-aa80841933e9
 - 1ababa8a-23b5-4db6-90c3-aa80841933e9\data_level0.bin
 - 1ababa8a-23b5-4db6-90c3-aa80841933e9\header.bin
 - 1ababa8a-23b5-4db6-90c3-aa80841933e9\length.bin
 - 1ababa8a-23b5-4db6-90c3-aa80841933e9\link_lists.bin
 - chroma.sqlite3
 - processed_chunks.jsonl


## Reload Database

To confirm persistence actually worked, we build a **new** `VectorDatabaseManager`
instance and load the collection back from disk rather than reusing the
in-memory object above.

In [8]:
reloaded_manager = VectorDatabaseManager(config, embedding_manager)
reloaded_store = reloaded_manager.load()

print("Reloaded vector count:", reloaded_manager.count())


2026-08-02 01:03:14 | INFO     | VectorDatabaseManager | Loading vector database from f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db
Reloaded vector count: 13


## Retriever Testing

A quick similarity search against the reloaded store to sanity-check that retrieval returns sensible, on-topic chunks.

In [9]:
test_query = "What is backpropagation used for?"
results = reloaded_store.similarity_search_with_relevance_scores(test_query, k=3)

for doc, score in results:
    meta = doc.metadata
    print(f"score={score:.3f} | {meta['course']} | {meta['file_name']} | page={meta.get('page_number')}")
    print("  ", doc.page_content[:150].replace("\n", " "))


score=0.490 | Deep Learning | neural_networks_basics.pdf | page=1
   Lecture 1: Neural Network Fundamentals A neural network is composed of layers of interconnected nodes, or neurons, each applying a weighted sum follow
score=0.158 | Deep Learning | glossary.txt | page=-1
   Deep Learning Glossary  Epoch: One complete pass through the entire training dataset during model training.  Dropout: A regularization technique that 
score=0.003 | Deep Learning | cnn_architectures.docx | page=-1
   Convolutional Neural Networks Convolutional Neural Networks (CNNs) are a class of deep neural networks primarily used for analyzing visual imagery. Th


## Similarity Search Examples

A few more queries spanning all three courses, to confirm the collection retrieves the right course's material for each topic.

In [10]:
example_queries = [
    "Explain the Turing Test.",
    "What is overfitting in machine learning?",
    "What does a convolutional neural network do?",
]

for query in example_queries:
    print(f"\nQuery: {query}")
    hits = reloaded_store.similarity_search_with_relevance_scores(query, k=2)
    for doc, score in hits:
        meta = doc.metadata
        print(f"  score={score:.3f} | {meta['course']} | {meta['file_name']}")



Query: Explain the Turing Test.
  score=0.397 | Artificial Intelligence | intro_to_ai.pdf
  score=-0.008 | Artificial Intelligence | glossary.txt

Query: What is overfitting in machine learning?
  score=0.653 | Machine Learning | glossary.txt
  score=0.215 | Deep Learning | glossary.txt

Query: What does a convolutional neural network do?
  score=0.550 | Deep Learning | cnn_architectures.docx
  score=0.336 | Deep Learning | neural_networks_basics.pdf


C:\Users\dell\AppData\Local\Temp\ipykernel_22884\790220082.py:9: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Artificial_Intelligence__intro_to_ai__p1__c00004', metadata={'file_name': 'intro_to_ai.pdf', 'doc_type': 'pdf', 'chunk_id': 'Artificial_Intelligence__intro_to_ai__p1__c00004', 'char_count': 613, 'page_number': 1, 'course': 'Artificial Intelligence'}, page_content='Lecture 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that typically require human intelligence, such as reasoning, learning, perception, and\nnatural language understanding.\nBranches of AI\nMajor branches of AI include Machine Learning, Natural Language Processing, Computer Vision,\nRobotics, Expert Systems, and Planning and Search.\nTuring Test\nThe Turing Test, proposed by Alan Turing in 1950, evaluates a machine ability to exhibit intelligent\nbehavior indistinguish

## Next Step

Continue to **`03_RAG_System.ipynb`** to wire up the retriever, prompt
template, and the Ollama LLM into the full RAG pipeline.